In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import joblib


print('Libraries loaded. Google Drive mounted at /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded. Google Drive mounted at /content/drive


In [ ]:
data_path = '/content/drive/MyDrive/expense_data_final_325.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f"CSV not found at: {data_path}\nCheck the filename in your Drive exactly.")
print("Found CSV at:", data_path)

# Load CSV
df = pd.read_csv(data_path)
print("Loaded data shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(10))

if 'Category' in df.columns:
    print("\nCategory distribution:\n", df['Category'].value_counts())
else:
    raise ValueError("CSV must contain a 'Category' column.")


Found CSV at: /content/drive/MyDrive/expense_data_final_325.csv
Loaded data shape: (300, 2)
Columns: ['Description', 'Category']


,Description,Category
0,YouTube Premium subscription #2,Entertainment
1,Paid for bike servicing #25,Services
2,Ordered premium pet food online #10,PetCare
3,Purchased antiseptic and bandages #5,Health
4,Lunch at cafe - sandwich and juice #21,Food
5,Airport shuttle bus ticket #23 at store #7,Transport
6,Bought greeting cards and gifts #24,Gifts
7,Bought board game for family #16 at store #14,Entertainment
8,Bought vitamins and supplements #20 - online,Health
9,Google Cloud storage renewal #17 at store #6,CloudServices



Category distribution:
 Category
Entertainment    25
Services         25
PetCare          25
Health           25
Food             25
Transport        25
Gifts            25
CloudServices    25
Household        25
Fashion          25
Groceries        25
Stationery       25
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = df.copy()
df['Description_clean'] = df['Description'].astype(str).str.lower().str.strip()

X = df['Description_clean']
y = df['Category']

# Stratified split (Safe now because every category has 25 samples)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape (train):", X_train_tfidf.shape)
print("TF-IDF shape (test):", X_test_tfidf.shape)


Train size: 225 Test size: 75
TF-IDF shape (train): (225, 926)
TF-IDF shape (test): (75, 926)


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Train
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# Save model and vectorizer
joblib.dump(model, "/content/drive/MyDrive/expense_model.joblib")
joblib.dump(vectorizer, "/content/drive/MyDrive/expense_vectorizer.joblib")

print("\nModel and vectorizer saved to Google Drive.")


Accuracy: 0.7733333333333333

Classification report:

               precision    recall  f1-score   support

CloudServices       0.56      0.83      0.67         6
Entertainment       0.67      1.00      0.80         6
      Fashion       1.00      0.83      0.91         6
         Food       1.00      0.43      0.60         7
        Gifts       1.00      1.00      1.00         6
    Groceries       0.62      0.83      0.71         6
       Health       0.86      0.86      0.86         7
    Household       1.00      0.43      0.60         7
      PetCare       1.00      1.00      1.00         6
     Services       0.60      0.50      0.55         6
   Stationery       0.60      1.00      0.75         6
    Transport       1.00      0.67      0.80         6

     accuracy                           0.77        75
    macro avg       0.83      0.78      0.77        75
 weighted avg       0.83      0.77      0.77        75


Model and vectorizer saved to Google Drive.


In [ ]:
def predict_expense(text):
    text_clean = text.lower().strip()
    vec = vectorizer.transform([text_clean])
    pred = model.predict(vec)[0]
    return pred

samples = [
    "Paid for Uber ride",
    "Bought vegetables from D-Mart",
    "Netflix subscription renewal",
    "Dog food purchase for puppy"
]

for s in samples:
    print(s, "→", predict_expense(s))


Paid for Uber ride → Services
Bought vegetables from D-Mart → Groceries
Netflix subscription renewal → Entertainment
Dog food purchase for puppy → PetCare


In [ ]:
import numpy as np

def predict_with_probs(text, top_k=3):
    txt = text.lower().strip()
    vec = vectorizer.transform([txt])
    probs = model.predict_proba(vec)[0]
    idx_sorted = np.argsort(probs)[::-1][:top_k]
    labels = model.classes_
    return [(labels[i], float(probs[i])) for i in idx_sorted]

samples = [
    "Paid for Uber ride",
    "Bought vegetables from D-Mart",
    "Netflix subscription renewal",
    "Dog food purchase for puppy"
]

for s in samples:
    print(s, "=>", predict_with_probs(s, top_k=3))


Paid for Uber ride => [(np.str_('Services'), 0.15722345011821812), (np.str_('Transport'), 0.14446738125977873), (np.str_('CloudServices'), 0.09964166548712605)]
Bought vegetables from D-Mart => [(np.str_('Groceries'), 0.14246045416156672), (np.str_('Gifts'), 0.10590647195269433), (np.str_('Fashion'), 0.09855070269197969)]
Netflix subscription renewal => [(np.str_('Entertainment'), 0.15533323530415355), (np.str_('CloudServices'), 0.15409399469703566), (np.str_('Services'), 0.07682612247901449)]
Dog food purchase for puppy => [(np.str_('PetCare'), 0.23373524865757653), (np.str_('Services'), 0.07587823690281449), (np.str_('Household'), 0.07504791434012836)]


In [ ]:
# ===== Combined: Calibration, Clean Predict, Threshold Analysis, Logging =====
import numpy as np
import pandas as pd
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1) Fit a calibrated classifier (Platt / sigmoid). Fast and robust.

calibrator = CalibratedClassifierCV(estimator=MultinomialNB(), cv=5, method='sigmoid')
calibrator.fit(X_train_tfidf, y_train)
print("Calibrator trained.")

# Save calibrator to Drive (so you can reload later)
calibrator_path = "/content/drive/MyDrive/expense_calibrator_sigmoid.joblib"
joblib.dump(calibrator, calibrator_path)
print("Calibrator saved to:", calibrator_path)

# 2) Clean, robust predict function that returns top-k and decision with threshold
def predict_calibrated_text(text, top_k=3, threshold=0.6):
    """
    Returns dict: { 'text', 'decision', 'topk': [(label,prob), ...] }
    decision == top label if top_prob >= threshold else "REVIEW"
    """
    txt = text.lower().strip()
    vec = vectorizer.transform([txt])
    probs = calibrator.predict_proba(vec)[0]
    idx_sorted = np.argsort(probs)[::-1][:top_k]
    labels = calibrator.classes_
    topk = [(str(labels[i]), float(probs[i])) for i in idx_sorted]
    top_label, top_prob = topk[0]
    decision = top_label if top_prob >= threshold else "REVIEW"
    return {"text": text, "decision": decision, "topk": topk, "top_prob": top_prob}

# Quick test
_examples = [
    "Paid for Uber ride",
    "Bought vegetables from D-Mart",
    "Netflix subscription renewal",
    "Dog food purchase for puppy"
]
print("\nSample calibrated predictions (threshold=0.6):")
for s in _examples:
    print(predict_calibrated_text(s, top_k=3, threshold=0.6))

# 3) Evaluate confidence distribution on the test set and help choose threshold
probs_test = calibrator.predict_proba(X_test_tfidf)
top_probs_test = probs_test.max(axis=1)

def show_confidence_stats(top_probs):
    print("\nConfidence stats (top-prob per sample):")
    print("Mean: %.3f  Median: %.3f  Min: %.3f  Max: %.3f" % (
        top_probs.mean(), np.median(top_probs), top_probs.min(), top_probs.max()))
    for p in [50, 60, 70, 80, 90, 95]:
        print(f"{p}th percentile: {np.percentile(top_probs, p):.3f}")

show_confidence_stats(top_probs_test)

# Table: performance @ different thresholds
print("\nThreshold analysis on test set (shows accepted_count and accuracy_on_accepted):")
for T in np.arange(0.50, 0.96, 0.05):
    mask = top_probs_test >= T
    accepted = mask.sum()
    if accepted == 0:
        acc = None
    else:
        preds = np.argmax(probs_test[mask], axis=1)
        acc = accuracy_score(y_test.to_numpy()[mask], preds)
    print(f"T={T:.2f}  accepted={accepted:3d}  accuracy_on_accepted={acc}")

# 4) Batch predict & log helper
def batch_predict_and_log(texts, threshold=0.6, outpath="/content/drive/MyDrive/predictions_calibrated_log.csv"):
    """
    texts: list of strings
    threshold: decision cutoff for auto-accept vs REVIEW
    outpath: path to save CSV on Drive
    Returns DataFrame and writes CSV
    """
    rows = []
    for t in texts:
        out = predict_calibrated_text(t, top_k=3, threshold=threshold)
        top1_label, top1_prob = out['topk'][0]
        second = out['topk'][1] if len(out['topk']) > 1 else ("", 0.0)
        third  = out['topk'][2] if len(out['topk']) > 2 else ("", 0.0)
        rows.append({
            "text": out['text'],
            "decision": out['decision'],
            "top1_label": top1_label,
            "top1_prob": top1_prob,
            "top2_label": second[0],
            "top2_prob": second[1],
            "top3_label": third[0],
            "top3_prob": third[1]
        })
    df_out = pd.DataFrame(rows)
    df_out.to_csv(outpath, index=False)
    print("Saved calibrated prediction log to:", outpath)
    return df_out


sample_inputs = _examples
log_df = batch_predict_and_log(sample_inputs, threshold=0.6,
                               outpath="/content/drive/MyDrive/sample_predictions_calibrated.csv")
log_df


Calibrator trained.
Calibrator saved to: /content/drive/MyDrive/expense_calibrator_sigmoid.joblib

Sample calibrated predictions (threshold=0.6):
{'text': 'Paid for Uber ride', 'decision': 'REVIEW', 'topk': [('Transport', 0.381143936982718), ('Services', 0.2123774218278355), ('CloudServices', 0.11266179171140242)], 'top_prob': 0.381143936982718}
{'text': 'Bought vegetables from D-Mart', 'decision': 'REVIEW', 'topk': [('Groceries', 0.5011748375559454), ('Fashion', 0.11212595683826672), ('Gifts', 0.08545914493996695)], 'top_prob': 0.5011748375559454}
{'text': 'Netflix subscription renewal', 'decision': 'REVIEW', 'topk': [('CloudServices', 0.3911988127227879), ('Entertainment', 0.37204452651506587), ('Fashion', 0.031432649695753836)], 'top_prob': 0.3911988127227879}
{'text': 'Dog food purchase for puppy', 'decision': 'PetCare', 'topk': [('PetCare', 0.6820765547759717), ('Fashion', 0.04064690328921026), ('Household', 0.04032173093735183)], 'top_prob': 0.6820765547759717}

Confidence stats 

TypeError: Labels in y_true and y_pred should be of the same type. Got y_true=['Services'] and y_pred=[9]. Make sure that the predictions provided by the classifier coincides with the true labels.

In [ ]:
# ===== Fixed threshold analysis & evaluation (use after calibrator is trained) =====
import numpy as np
from sklearn.metrics import accuracy_score
import pandas as pd


print("probs_test shape:", probs_test.shape, "y_test shape:", y_test.shape)

label_names = list(calibrator.classes_)
y_true = y_test.to_numpy()

rows = []
for T in np.arange(0.50, 0.96, 0.05):
    mask = top_probs_test >= T
    accepted = int(mask.sum())
    if accepted == 0:
        acc = None
    else:
        preds_idx = np.argmax(probs_test[mask], axis=1)
        preds_labels = [label_names[i] for i in preds_idx]
        acc = accuracy_score(y_true[mask], preds_labels)
    rows.append({"threshold": float(T), "accepted": accepted, "accuracy_on_accepted": acc})

df_thresh = pd.DataFrame(rows)
print(df_thresh.to_string(index=False))


print("\nRecommended thresholds (accepted>0):")
print(df_thresh[df_thresh['accepted']>0].sort_values(by='accuracy_on_accepted', ascending=False).head(10))


probs_test shape: (75, 12) y_test shape: (75,)
 threshold  accepted  accuracy_on_accepted
      0.50        39                   1.0
      0.55        31                   1.0
      0.60        26                   1.0
      0.65        20                   1.0
      0.70        16                   1.0
      0.75         6                   1.0
      0.80         1                   1.0
      0.85         0                   NaN
      0.90         0                   NaN
      0.95         0                   NaN

Recommended thresholds (accepted>0):
   threshold  accepted  accuracy_on_accepted
0       0.50        39                   1.0
1       0.55        31                   1.0
2       0.60        26                   1.0
3       0.65        20                   1.0
4       0.70        16                   1.0
5       0.75         6                   1.0
6       0.80         1                   1.0


In [ ]:
import os, pandas as pd, numpy as np

drive_folder = "/content/drive/MyDrive"
print("Listing files in MyDrive (top 200):")
for i,f in enumerate(sorted(os.listdir(drive_folder))[:200]):
    print(i+1, f)
print("\n-- Checking specific files --")

expected1 = os.path.join(drive_folder, "predictions_calibrated_log.csv")
expected2 = os.path.join(drive_folder, "sample_predictions_calibrated.csv")
expected3 = os.path.join(drive_folder, "sample_predictions_calibrated.csv")

for p in [expected1, expected2]:
    print(p, "->", "EXISTS" if os.path.exists(p) else "MISSING")


try:

    print("\nRecreating logs now...")
    sample_inputs = [
        "Paid for Uber ride",
        "Bought vegetables from D-Mart",
        "Netflix subscription renewal",
        "Dog food purchase for puppy"
    ]
    rows = []
    for t in sample_inputs:
        out = predict_calibrated_text(t, top_k=3, threshold=0.6)
        top1_label, top1_prob = out['topk'][0]
        second = out['topk'][1] if len(out['topk'])>1 else ("",0.0)
        third  = out['topk'][2] if len(out['topk'])>2 else ("",0.0)
        rows.append({
            "text": out['text'],
            "decision": out['decision'],
            "top1_label": top1_label,
            "top1_prob": top1_prob,
            "top2_label": second[0],
            "top2_prob": second[1],
            "top3_label": third[0],
            "top3_prob": third[1]
        })
    df_log = pd.DataFrame(rows)
    df_log.to_csv(expected1, index=False)
    df_log.to_csv(expected2, index=False)
    print("Wrote:", expected1)
    print("Wrote:", expected2)
    display(df_log)
except Exception as e:
    print("Could not recreate logs automatically. Error:", e)
    print("Make sure `calibrator` and `predict_calibrated_text` are in memory and rerun the combined calibration cell first.")


Listing files in MyDrive (top 200):
1 .ipynb_checkpoints
2 Classroom
3 Colab Notebooks
4 Frontend.zip
5 GenAI_certificate.pdf
6 NPTELCERTIFICATE.pdf
7 Orchathon_2K25_Round_1_Result.pdf
8 Prompts.docx
9 Prompts.gdoc
10 Resume.gdoc
11 Share Orchathon certificate.pdf
12 Share java_basic certificate.pdf
13 Tickets - Odoo Hackathon 2025 (11 Aug 2025, 08_00_00).pdf
14 Untitled Diagram.drawio
15 Untitled document (1).gdoc
16 Untitled document (2).gdoc
17 Untitled document (3).gdoc
18 Untitled document.gdoc
19 VID-20250712-WA0045.mp4
20 expense_calibrator_sigmoid.joblib
21 expense_data_final_325.csv
22 expense_model.joblib
23 expense_vectorizer.joblib
24 explain big data nosql and cap theorem.gsheet
25 hackerrankjava.jpg
26 mypic.pdf
27 myresume.docx
28 myresume.pdf
29 passportsizephoto.pdf
30 photo4.jpg
31 sheetal.docx
32 smart-life-care.zip
33 tenthmc.pdf
34 voterId.pdf
35 “CulturaX – The Emotionally Intelligent Cultural Support Bot”.gdoc

-- Checking specific files --
/content/drive/MyDrive

,text,decision,top1_label,top1_prob,top2_label,top2_prob,top3_label,top3_prob
0,Paid for Uber ride,REVIEW,Transport,0.381144,Services,0.212377,CloudServices,0.112662
1,Bought vegetables from D-Mart,REVIEW,Groceries,0.501175,Fashion,0.112126,Gifts,0.085459
2,Netflix subscription renewal,REVIEW,CloudServices,0.391199,Entertainment,0.372045,Fashion,0.031433
3,Dog food purchase for puppy,PetCare,PetCare,0.682077,Fashion,0.040647,Household,0.040322


In [ ]:
# ===== Runtime helpers: calibrated predict + batch logging =====
import pandas as pd
import joblib
import os
from typing import List, Dict


T_PROD = 0.60


def predict_calibrated_text(text: str, top_k: int = 3, threshold: float = T_PROD) -> Dict:
    """
    Predict single text using calibrated classifier.
    Returns dictionary:
      { text, decision, topk: [(label, prob), ...], top_prob }
    decision == top label if top_prob >= threshold else "REVIEW"
    """
    txt = text.lower().strip()
    vec = vectorizer.transform([txt])
    probs = calibrator.predict_proba(vec)[0]
    idx_sorted = probs.argsort()[::-1][:top_k]
    labels = list(calibrator.classes_)
    topk = [(labels[i], float(probs[i])) for i in idx_sorted]
    top_label, top_prob = topk[0]
    decision = top_label if top_prob >= threshold else "REVIEW"
    return {"text": text, "decision": decision, "topk": topk, "top_prob": float(top_prob)}

def batch_predict_and_log(texts: List[str],
                          threshold: float = T_PROD,
                          outpath: str = "/content/drive/MyDrive/predictions_calibrated_log.csv") -> pd.DataFrame:
    """
    Batch predict a list of texts, apply threshold, and save a CSV to Drive.
    Returns the DataFrame that was saved.
    Columns: text, decision, top1_label, top1_prob, top2_label, top2_prob, top3_label, top3_prob
    """
    rows = []
    for t in texts:
        out = predict_calibrated_text(t, top_k=3, threshold=threshold)
        top1_label, top1_prob = out['topk'][0]
        top2_label, top2_prob = out['topk'][1] if len(out['topk'])>1 else ("", 0.0)
        top3_label, top3_prob = out['topk'][2] if len(out['topk'])>2 else ("", 0.0)
        rows.append({
            "text": out['text'],
            "decision": out['decision'],
            "top1_label": top1_label,
            "top1_prob": top1_prob,
            "top2_label": top2_label,
            "top2_prob": top2_prob,
            "top3_label": top3_label,
            "top3_prob": top3_prob
        })
    df_out = pd.DataFrame(rows)

    os.makedirs(os.path.dirname(outpath), exist_ok=True)
    df_out.to_csv(outpath, index=False)
    print("Saved calibrated prediction log to:", outpath)
    return df_out


sample_inputs = [
    "Paid for Uber ride",
    "Bought vegetables from D-Mart",
    "Netflix subscription renewal",
    "Dog food purchase for puppy"
]

# Single prediction example
print("Single example:", predict_calibrated_text(sample_inputs[0], top_k=3, threshold=T_PROD))

# Batch predict + save to Drive
df_log = batch_predict_and_log(sample_inputs, threshold=T_PROD,
                               outpath="/content/drive/MyDrive/runtime_predictions_calibrated.csv")
df_log.head()


Single example: {'text': 'Paid for Uber ride', 'decision': 'REVIEW', 'topk': [('Transport', 0.381143936982718), ('Services', 0.2123774218278355), ('CloudServices', 0.11266179171140242)], 'top_prob': 0.381143936982718}
Saved calibrated prediction log to: /content/drive/MyDrive/runtime_predictions_calibrated.csv


,text,decision,top1_label,top1_prob,top2_label,top2_prob,top3_label,top3_prob
0,Paid for Uber ride,REVIEW,Transport,0.381144,Services,0.212377,CloudServices,0.112662
1,Bought vegetables from D-Mart,REVIEW,Groceries,0.501175,Fashion,0.112126,Gifts,0.085459
2,Netflix subscription renewal,REVIEW,CloudServices,0.391199,Entertainment,0.372045,Fashion,0.031433
3,Dog food purchase for puppy,PetCare,PetCare,0.682077,Fashion,0.040647,Household,0.040322


In [ ]:

import pandas as pd
from google.colab import files
import os

# configure
INPUT_CSV = "/content/drive/MyDrive/expense_data_final_325.csv"
OUTPUT_CSV = "/content/drive/MyDrive/runtime_predictions_calibrated_full.csv"
THRESH = 0.60

df = pd.read_csv(INPUT_CSV)
texts = df['Description'].astype(str).tolist()


rows = []
for t in texts:
    out = predict_calibrated_text(t, top_k=3, threshold=THRESH)
    top1_label, top1_prob = out['topk'][0]
    top2_label, top2_prob = out['topk'][1] if len(out['topk'])>1 else ("", 0.0)
    top3_label, top3_prob = out['topk'][2] if len(out['topk'])>2 else ("", 0.0)
    rows.append({
        "Description": out['text'],
        "decision": out['decision'],
        "top1_label": top1_label,
        "top1_prob": top1_prob,
        "top2_label": top2_label,
        "top2_prob": top2_prob,
        "top3_label": top3_label,
        "top3_prob": top3_prob
    })

df_out = pd.DataFrame(rows)
df_out.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

# Summary stats
accepted = (df_out['decision'] != "REVIEW").sum()
reviewed = (df_out['decision'] == "REVIEW").sum()
print(f"Total rows: {len(df_out)}  Accepted: {accepted}  REVIEW: {reviewed}")
display(df_out.head(20))


files.download(OUTPUT_CSV)


Saved: /content/drive/MyDrive/runtime_predictions_calibrated_full.csv
Total rows: 300  Accepted: 195  REVIEW: 105


,Description,decision,top1_label,top1_prob,top2_label,top2_prob,top3_label,top3_prob
0,YouTube Premium subscription #2,Entertainment,Entertainment,0.650067,CloudServices,0.057693,Health,0.049429
1,Paid for bike servicing #25,REVIEW,Services,0.569165,CloudServices,0.091960,Food,0.080335
2,Ordered premium pet food online #10,PetCare,PetCare,0.661890,Health,0.067043,Food,0.063902
3,Purchased antiseptic and bandages #5,Health,Health,0.703064,Household,0.051103,Stationery,0.037381
4,Lunch at cafe - sandwich and juice #21,Food,Food,0.801299,Fashion,0.026529,Stationery,0.021906
5,Airport shuttle bus ticket #23 at store #7,REVIEW,Transport,0.578791,Entertainment,0.087369,Groceries,0.056807
6,Bought greeting cards and gifts #24,Gifts,Gifts,0.719025,Stationery,0.084780,Fashion,0.032730
7,Bought board game for family #16 at store #14,REVIEW,Entertainment,0.550837,Food,0.081521,PetCare,0.057278
8,Bought vitamins and supplements #20 - online,Health,Health,0.739717,Stationery,0.044046,Household,0.033020
9,Google Cloud storage renewal #17 at store #6,CloudServices,CloudServices,0.686959,Food,0.051034,Groceries,0.044043


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

main_csv = "/content/drive/MyDrive/expense_data_final_325.csv"
corrected_clean_csv = "/content/drive/MyDrive/expense_data_revieww_corrected.csv"
augmented_out = "/content/drive/MyDrive/expense_data_augmented.csv"

df_main = pd.read_csv(main_csv)
df_fixed = pd.read_csv(corrected_clean_csv)

# Combine files
df_augmented = pd.concat([df_main, df_fixed], ignore_index=True)

# Save
df_augmented.to_csv(augmented_out, index=False)

print("Appended successfully.")
print("Original size:", len(df_main))
print("Corrected rows:", len(df_fixed))
print("New training size:", len(df_augmented))
print("Saved new training CSV to:", augmented_out)


Appended successfully.
Original size: 300
Corrected rows: 300
New training size: 600
Saved new training CSV to: /content/drive/MyDrive/expense_data_augmented.csv


In [ ]:
#  Retrain on augmented data (Cell)
import pandas as pd, joblib, os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

DRIVE = "/content/drive/MyDrive"
DATA_PATH = Path(DRIVE) / "expense_data_augmented.csv"
MODEL_OUT = Path(DRIVE) / "expense_model_retrained.joblib"
VECT_OUT  = Path(DRIVE) / "expense_vectorizer_retrained.joblib"

df = pd.read_csv(DATA_PATH)
print("Loaded augmented data shape:", df.shape)

# Ensure columns
assert 'Description' in df.columns and 'Category' in df.columns, "CSV must have Description and Category"

df['Description_clean'] = df['Description'].astype(str).str.lower().str.strip()
X = df['Description_clean']
y = df['Category']

# Stratified split 75/25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train size:", len(X_train), "Test size:", len(X_test))


vectorizer = TfidfVectorizer(ngram_range=(1,3), min_df=2, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print("TF-IDF shapes -> train:", X_train_tfidf.shape, " test:", X_test_tfidf.shape)

# Train NB
model = MultinomialNB(alpha=1.0)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print("Retrained model accuracy:", acc)
print("\nClassification report:\n", classification_report(y_test, y_pred, zero_division=0))

# Save artifacts
joblib.dump(model, MODEL_OUT)
joblib.dump(vectorizer, VECT_OUT)
print("Saved model to:", MODEL_OUT)
print("Saved vectorizer to:", VECT_OUT)


Loaded augmented data shape: (600, 2)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
# Inspect counts and back up current augmented CSV
import pandas as pd
from pathlib import Path
DRIVE = "/content/drive/MyDrive"
AUG_CSV = Path(DRIVE) / "expense_data_augmented.csv"
BACKUP = Path(DRIVE) / "expense_data_augmented_backup_before_fix.csv"

df_aug = pd.read_csv(AUG_CSV)
print("Loaded augmented data shape:", df_aug.shape)
counts = df_aug['Category'].value_counts()
print("\nCategory counts:\n", counts)

# Backup
df_aug.to_csv(BACKUP, index=False)
print("\nBackup saved to:", BACKUP)


Loaded augmented data shape: (600, 2)

Category counts:
 Category
Services         56
Gifts            52
Health           50
CloudServices    50
PetCare          50
Groceries        50
Fashion          50
Stationery       49
Food             49
Household        49
Transport        48
Entertainment    46
REVIEW            1
Name: count, dtype: int64

Backup saved to: /content/drive/MyDrive/expense_data_augmented_backup_before_fix.csv


In [ ]:
# Duplicate rows for classes with count < 2 so stratify will work
from collections import Counter
import random

df = df_aug.copy()
counts = df['Category'].value_counts()
print("Before fix, min count:", counts.min())

# Find classes with only 1 sample
rare = counts[counts < 2].index.tolist()
print("Classes with <2 samples (will be duplicated):", rare)

# For each rare class, duplicate random rows from that class until count >= 2
for cls in rare:
    rows_cls = df[df['Category'] == cls]
    if len(rows_cls) == 0:
        continue
    while df['Category'].value_counts().get(cls,0) < 2:
        chosen = rows_cls.sample(n=1, replace=True, random_state=random.randint(0,9999))
        df = pd.concat([df, chosen], ignore_index=True)

# Verify
new_counts = df['Category'].value_counts()
print("After fix, min count:", new_counts.min())
print(new_counts)

# Save fixed CSV
FIXED_CSV = Path(DRIVE) / "expense_data_augmented_fixed_for_stratify.csv"
df.to_csv(FIXED_CSV, index=False)
print("Saved fixed file to:", FIXED_CSV)


Before fix, min count: 1
Classes with <2 samples (will be duplicated): ['REVIEW']
After fix, min count: 2
Category
Services         56
Gifts            52
Health           50
CloudServices    50
PetCare          50
Groceries        50
Fashion          50
Stationery       49
Food             49
Household        49
Transport        48
Entertainment    46
REVIEW            2
Name: count, dtype: int64
Saved fixed file to: /content/drive/MyDrive/expense_data_augmented_fixed_for_stratify.csv


In [ ]:
# Retrain on fixed augmented data
import pandas as pd, joblib, os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

DRIVE = "/content/drive/MyDrive"
DATA_PATH = Path(DRIVE) / "expense_data_augmented_fixed_for_stratify.csv"
MODEL_OUT = Path(DRIVE) / "expense_model_retrained.joblib"
VECT_OUT  = Path(DRIVE) / "expense_vectorizer_retrained.joblib"

df = pd.read_csv(DATA_PATH)
print("Loaded fixed augmented data shape:", df.shape)

# Ensure columns
assert 'Description' in df.columns and 'Category' in df.columns, "CSV must have Description and Category"

df['Description_clean'] = df['Description'].astype(str).str.lower().str.strip()
X = df['Description_clean']
y = df['Category']

# Stratified split 75/25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train size:", len(X_train), "Test size:", len(X_test))

# Recommended vectorizer (good balance for performance)
vectorizer = TfidfVectorizer(ngram_range=(1,3), min_df=2, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print("TF-IDF shapes -> train:", X_train_tfidf.shape, " test:", X_test_tfidf.shape)

# Train NB
model = MultinomialNB(alpha=1.0)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print("Retrained model accuracy:", acc)
print("\nClassification report:\n", classification_report(y_test, y_pred, zero_division=0))

# Save artifacts
joblib.dump(model, MODEL_OUT)
joblib.dump(vectorizer, VECT_OUT)
print("Saved model to:", MODEL_OUT)
print("Saved vectorizer to:", VECT_OUT)


Loaded fixed augmented data shape: (601, 2)
Train size: 450 Test size: 151
TF-IDF shapes -> train: (450, 1331)  test: (151, 1331)
Retrained model accuracy: 0.8874172185430463

Classification report:
                precision    recall  f1-score   support

CloudServices       0.85      0.85      0.85        13
Entertainment       1.00      0.91      0.95        11
      Fashion       0.71      0.77      0.74        13
         Food       0.92      0.92      0.92        12
        Gifts       0.76      1.00      0.87        13
    Groceries       1.00      0.85      0.92        13
       Health       1.00      0.69      0.82        13
    Household       1.00      0.92      0.96        12
      PetCare       1.00      1.00      1.00        13
     Services       0.74      1.00      0.85        14
   Stationery       0.92      1.00      0.96        12
    Transport       1.00      0.75      0.86        12

     accuracy                           0.89       151
    macro avg       0.91    

In [ ]:
#  Recalibrate with CalibratedClassifierCV and analyze thresholds
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# First, fit the base estimator (MultinomialNB)
base_estimator = MultinomialNB()
base_estimator.fit(X_train_tfidf, y_train)


calibrator = CalibratedClassifierCV(estimator=base_estimator, cv='prefit', method='sigmoid')
calibrator.fit(X_train_tfidf, y_train)

cal_path = Path(DRIVE) / "expense_calibrator_sigmoid_retrained.joblib"
joblib.dump(calibrator, cal_path)
print("Calibrator trained and saved:", cal_path)

# Threshold analysis
probs_test = calibrator.predict_proba(X_test_tfidf)
top_probs_test = probs_test.max(axis=1)
label_names = list(calibrator.classes_)
y_true = y_test.to_numpy()

rows = []
for T in np.arange(0.50, 0.96, 0.05):
    mask = top_probs_test >= T
    accepted = int(mask.sum())
    if accepted == 0:
        acc = None
    else:
        preds_idx = np.argmax(probs_test[mask], axis=1)
        preds_labels = [label_names[i] for i in preds_idx]
        acc = accuracy_score(y_true[mask], preds_labels)
    rows.append({"threshold": float(T), "accepted": accepted, "accuracy_on_accepted": acc})

df_thresh = pd.DataFrame(rows)
print(df_thresh.to_string(index=False))

Calibrator trained and saved: /content/drive/MyDrive/expense_calibrator_sigmoid_retrained.joblib
 threshold  accepted  accuracy_on_accepted
      0.50       130              0.969231
      0.55       128              0.984375
      0.60       127              0.984252
      0.65       125              0.984000
      0.70       123              0.983740
      0.75       121              0.983471
      0.80       117              0.991453
      0.85       113              1.000000
      0.90        97              1.000000
      0.95        23              1.000000


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [ ]:
# Set production threshold (choose 0.60 or 0.65)
T_PROD = 0.60

# Quick sanity print
print("Production threshold set to:", T_PROD)

probs_test = calibrator.predict_proba(X_test_tfidf)
accepted = (probs_test.max(axis=1) >= T_PROD).sum()
print("Expected accepted on test set (approx):", int(accepted), "out of", len(probs_test))


Production threshold set to: 0.6
Expected accepted on test set (approx): 127 out of 151


In [ ]:
#  Run calibrated predictions over the working CSV and save using production threshold
import pandas as pd
import joblib
import os
from typing import List, Dict

# Production threshold
T_PROD = 0.60


def predict_calibrated_text(text: str, top_k: int = 3, threshold: float = T_PROD) -> Dict:
    """
    Predict single text using calibrated classifier.
    Returns dictionary:
      { text, decision, topk: [(label, prob), ...], top_prob }
    decision == top label if top_prob >= threshold else "REVIEW"
    """
    txt = text.lower().strip()

    vec = vectorizer.transform([txt])
    probs = calibrator.predict_proba(vec)[0]
    idx_sorted = probs.argsort()[::-1][:top_k]
    labels = list(calibrator.classes_)
    topk = [(labels[i], float(probs[i])) for i in idx_sorted]
    top_label, top_prob = topk[0]
    decision = top_label if top_prob >= threshold else "REVIEW"
    return {"text": text, "decision": decision, "topk": topk, "top_prob": float(top_prob)}

def batch_predict_and_log(texts: List[str],
                          threshold: float = T_PROD,
                          outpath: str = "/content/drive/MyDrive/predictions_calibrated_log.csv") -> pd.DataFrame:
    """
    Batch predict a list of texts, apply threshold, and save a CSV to Drive.
    Returns the DataFrame that was saved.
    Columns: text, decision, top1_label, top1_prob, top2_label, top2_prob, top3_label, top3_prob
    """
    rows = []
    for t in texts:
        out = predict_calibrated_text(t, top_k=3, threshold=threshold)
        top1_label, top1_prob = out['topk'][0]
        top2_label, top2_prob = out['topk'][1] if len(out['topk'])>1 else ("", 0.0)
        top3_label, top3_prob = out['topk'][2] if len(out['topk'])>2 else ("", 0.0)
        rows.append({
            "text": out['text'],
            "decision": out['decision'],
            "top1_label": top1_label,
            "top1_prob": top1_prob,
            "top2_label": top2_label,
            "top2_prob": top2_prob,
            "top3_label": top3_label,
            "top3_prob": top3_prob
        })
    df_out = pd.DataFrame(rows)
    # Ensure Drive dir exists
    os.makedirs(os.path.dirname(outpath), exist_ok=True)
    df_out.to_csv(outpath, index=False)
    print("Saved calibrated prediction log to:", outpath)
    return df_out

INPUT_CSV = "/content/drive/MyDrive/expense_data_augmented_fixed_for_stratify.csv"  # or final dataset path you want to process
OUT_CSV = "/content/drive/MyDrive/runtime_predictions_calibrated_prod.csv"

df_in = pd.read_csv(INPUT_CSV)
rows_df = batch_predict_and_log(df_in['Description'].astype(str).tolist(), threshold=T_PROD, outpath=OUT_CSV)
print("Saved runtime production log to:", OUT_CSV)
print("Accepted:", (rows_df['decision'] != "REVIEW").sum(), " Review:", (rows_df['decision']=="REVIEW").sum())

Saved calibrated prediction log to: /content/drive/MyDrive/runtime_predictions_calibrated_prod.csv
Saved runtime production log to: /content/drive/MyDrive/runtime_predictions_calibrated_prod.csv
Accepted: 556  Review: 45


In [ ]:
# Debug test
test_text = "ordered burger from swiggy."

try:

    import joblib, os
    from pathlib import Path
    DRIVE = Path("/content/drive/MyDrive")
    if 'vectorizer' not in globals():
        vectorizer = joblib.load(str(DRIVE / "expense_vectorizer_retrained.joblib"))
    if 'calibrator' not in globals():
        calibrator = joblib.load(str(DRIVE / "expense_calibrator_sigmoid_retrained.joblib"))

    def predict_calibrated_text_local(text: str, top_k: int = 3, threshold: float = 0.60):
        txt = text.lower().strip()
        vec = vectorizer.transform([txt])
        probs = calibrator.predict_proba(vec)[0]
        idx_sorted = probs.argsort()[::-1][:top_k]
        label_names = list(calibrator.classes_)
        topk = [(label_names[i], float(probs[i])) for i in idx_sorted]
        top_label, top_prob = topk[0]
        decision = top_label if top_prob >= threshold else "REVIEW"
        return {"text": text, "decision": decision, "topk": topk, "top_prob": float(top_prob)}
    out = predict_calibrated_text_local(test_text, top_k=3, threshold=0.60)
    print("PREDICTION OK:", out)
except Exception as e:
    import traceback, sys
    print("ERROR during direct prediction test:")
    traceback.print_exc(limit=20)


PREDICTION OK: {'text': 'ordered burger from swiggy.', 'decision': np.str_('Food'), 'topk': [(np.str_('Food'), 0.6239208172860998), (np.str_('Fashion'), 0.06401684153247575), (np.str_('Stationery'), 0.05098951977609482)], 'top_prob': 0.6239208172860998}


In [ ]:
# Robust Gradio UI:
!pip install -q gradio >/dev/null

import gradio as gr
import joblib, pandas as pd, os, datetime
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
LOG_PATH = DRIVE / "expense_entries_log.csv"


if 'vectorizer' not in globals():
    vectorizer = joblib.load(str(DRIVE / "expense_vectorizer_retrained.joblib"))
if 'calibrator' not in globals():
    calibrator = joblib.load(str(DRIVE / "expense_calibrator_sigmoid_retrained.joblib"))
label_names = list(calibrator.classes_)

def predict_action(text: str, amount, threshold: float):
    """Return explicit fields and a state dict to save later."""
    try:
        if not text or str(text).strip()=="":
            return ("", "", "", "", 0.0, "Enter a description." , {})
        txt = text.lower().strip()
        vec = vectorizer.transform([txt])
        probs = calibrator.predict_proba(vec)[0]
        idx = probs.argsort()[::-1][:3]
        topk = [(label_names[i], float(probs[i])) for i in idx]
        top1_str = f"{topk[0][0]} ({topk[0][1]:.3f})"
        top2_str = f"{topk[1][0]} ({topk[1][1]:.3f})" if len(topk)>1 else ""
        top3_str = f"{topk[2][0]} ({topk[2][1]:.3f})" if len(topk)>2 else ""
        top_prob = float(topk[0][1])
        decision = topk[0][0] if top_prob >= threshold else "REVIEW"

        state = {
            "text": text,
            "amount": amount,
            "topk": topk,
            "decision": decision,
            "predicted_label": topk[0][0],
            "predicted_confidence": topk[0][1]
        }
        return (decision, top1_str, top2_str, top3_str, round(top_prob,4), "Prediction OK", state)
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return ("", "", "", "", 0.0, f"Prediction error: {str(e)}\n{tb}", {})

def save_action(selected_label, state):
    """Save the last prediction/state to Drive, using selected_label as corrected label if provided."""
    try:
        if not state or "text" not in state:
            return ("No prediction found — run Predict first.", {})
        ts = datetime.datetime.now().isoformat()
        row = {
            "timestamp": ts,
            "description": state.get("text",""),
            "amount": state.get("amount", None),
            "predicted_label": state.get("predicted_label",""),
            "predicted_confidence": float(state.get("predicted_confidence", 0.0)),
            "decision": state.get("decision",""),
            "corrected_label": selected_label if selected_label else ""
        }

        if LOG_PATH.exists():
            df = pd.read_csv(LOG_PATH)
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        else:
            df = pd.DataFrame([row])
        df.to_csv(LOG_PATH, index=False)
        return ("Saved entry to Drive.", row)
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return (f"Save error: {str(e)}\n{tb}", {})


with gr.Blocks() as demo:
    gr.Markdown("## Expense Categorizer — Stable UI\nPredict, correct, and save expenses to Drive.")
    with gr.Row():
        with gr.Column(scale=3):
            desc = gr.Textbox(label="Description", placeholder="e.g. ordered burger from swiggy.", lines=2)
            amt = gr.Number(label="Amount (optional)", value=None)
            thr = gr.Slider(label="Confidence threshold (>= accepted)", minimum=0.1, maximum=0.99, value=0.60, step=0.01)
            with gr.Row():
                predict_btn = gr.Button("Predict")
                save_btn = gr.Button("Save")
        with gr.Column(scale=2):
            out_dec = gr.Textbox(label="Decision", interactive=False)
            out_top1 = gr.Textbox(label="Top-1 (label (prob))", interactive=False)
            out_top2 = gr.Textbox(label="Top-2", interactive=False)
            out_top3 = gr.Textbox(label="Top-3", interactive=False)
            out_conf = gr.Number(label="Top-1 confidence", interactive=False)
            corrected = gr.Dropdown(label="Corrected Label (optional)", choices=label_names, value=None)
            status = gr.Textbox(label="Status / Errors", interactive=False)


    state = gr.State({})


    predict_btn.click(fn=predict_action, inputs=[desc, amt, thr],
                      outputs=[out_dec, out_top1, out_top2, out_top3, out_conf, status, state])


    save_btn.click(fn=save_action, inputs=[corrected, state], outputs=[status, gr.JSON()])

    gr.Markdown(f"Saved entries will be appended to: `{LOG_PATH}`")

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://772080b60c2d712308.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
# Gradio app with embedded interactive Dashboard (Plotly)
!pip install -q gradio plotly >/dev/null

import gradio as gr, joblib, pandas as pd, os, datetime
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
LOG_PATH = DRIVE / "expense_entries_log.csv"

# --- Load ML artifacts safely ---
if 'vectorizer' not in globals():
    vectorizer = joblib.load(str(DRIVE / "expense_vectorizer_retrained.joblib"))
if 'calibrator' not in globals():
    calibrator = joblib.load(str(DRIVE / "expense_calibrator_sigmoid_retrained.joblib"))
label_names = list(calibrator.classes_)

# --- Predict & Save functions (robust) ---
def predict_action(text: str, amount, threshold: float):
    try:
        if not text or str(text).strip()=="":
            return ("", "", "", "", 0.0, "Enter a description." , {})
        txt = text.lower().strip()
        vec = vectorizer.transform([txt])
        probs = calibrator.predict_proba(vec)[0]
        idx = probs.argsort()[::-1][:3]
        topk = [(label_names[i], float(probs[i])) for i in idx]
        top1_str = f"{topk[0][0]} ({topk[0][1]:.3f})"
        top2_str = f"{topk[1][0]} ({topk[1][1]:.3f})" if len(topk)>1 else ""
        top3_str = f"{topk[2][0]} ({topk[2][1]:.3f})" if len(topk)>2 else ""
        top_prob = float(topk[0][1])
        decision = topk[0][0] if top_prob >= threshold else "REVIEW"
        state = {
            "text": text,
            "amount": amount,
            "topk": topk,
            "decision": decision,
            "predicted_label": topk[0][0],
            "predicted_confidence": topk[0][1]
        }
        return (decision, top1_str, top2_str, top3_str, round(top_prob,4), "Prediction OK", state)
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return ("", "", "", "", 0.0, f"Prediction error: {str(e)}\n{tb}", {})

def save_action(selected_label, state):
    try:
        if not state or "text" not in state:
            return ("No prediction found — run Predict first.", {})
        ts = datetime.datetime.now().isoformat()
        row = {
            "timestamp": ts,
            "description": state.get("text",""),
            "amount": state.get("amount", None),
            "predicted_label": state.get("predicted_label",""),
            "predicted_confidence": float(state.get("predicted_confidence", 0.0)),
            "decision": state.get("decision",""),
            "corrected_label": selected_label if selected_label else ""
        }
        if LOG_PATH.exists():
            df = pd.read_csv(LOG_PATH)
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        else:
            df = pd.DataFrame([row])
        df.to_csv(LOG_PATH, index=False)
        return ("Saved entry to Drive.", row)
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        return (f"Save error: {str(e)}\n{tb}", {})

# --- Dashboard
def load_log_df():
    if not LOG_PATH.exists():
        return pd.DataFrame(columns=["timestamp","description","amount","predicted_label","predicted_confidence","decision","corrected_label"])
    df = pd.read_csv(LOG_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce').fillna(pd.Timestamp.now())
    # normalize amount
    def parse_amount(x):
        try:
            if pd.isna(x): return 0.0
            s = str(x)
            for ch in ['₹','$','Rs','rs','INR',',']:
                s = s.replace(ch,'')
            return float(s) if s.strip()!="" else 0.0
        except:
            import re
            m = re.findall(r"[-+]?[0-9]*\.?[0-9]+", str(x))
            return float(m[0]) if m else 0.0
    df['amount_clean'] = df.get('amount', 0).apply(parse_amount)
    # final category preference: corrected_label if exists else predicted_label else decision
    df['category_final'] = df.apply(lambda r: r['corrected_label'] if pd.notna(r.get('corrected_label')) and str(r.get('corrected_label')).strip()!="" else (r.get('predicted_label') if pd.notna(r.get('predicted_label')) else r.get('decision')), axis=1)
    df['year_month'] = df['timestamp'].dt.to_period('M').astype(str)
    return df

def make_monthly_fig(df):
    monthly = df.groupby('year_month')['amount_clean'].sum().reset_index().sort_values('year_month')
    if monthly.empty:
        fig = go.Figure().update_layout(title="No data")
        return fig
    fig = px.bar(monthly, x='year_month', y='amount_clean', title="Total Spend by Month", labels={'year_month':'Month','amount_clean':'Amount'})
    fig.update_xaxes(tickangle= -45)
    return fig

def make_category_fig(df):
    by_cat = df.groupby('category_final')['amount_clean'].sum().reset_index().sort_values('amount_clean', ascending=False)
    if by_cat.empty:
        return go.Figure().update_layout(title="No data")
    fig = px.bar(by_cat, x='amount_clean', y='category_final', orientation='h', title="Total Spend by Category", labels={'amount_clean':'Amount','category_final':'Category'})
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    return fig

def make_heatmap_fig(df):
    pivot = df.pivot_table(index='category_final', columns='year_month', values='amount_clean', aggfunc='sum', fill_value=0)
    if pivot.empty:
        return go.Figure().update_layout(title="No data")
    # sort columns
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='Viridis'
    ))
    fig.update_layout(title="Category spend per Month (heatmap)", xaxis_title="Month", yaxis_title="Category")
    return fig

def build_dashboard():
    df = load_log_df()
    fig_month = make_monthly_fig(df)
    fig_cat = make_category_fig(df)
    fig_heat = make_heatmap_fig(df)
    table_df = df[['timestamp','description','amount_clean','category_final']].sort_values('timestamp', ascending=False).head(200)
    table_df = table_df.rename(columns={'amount_clean':'amount','category_final':'category'})
    return fig_month, fig_cat, fig_heat, table_df

# --- Gradio UI with tabs ---
with gr.Blocks(title="Expense Categorizer + Dashboard") as app:
    gr.Markdown("## Expense Categorizer — Predict, Save, and Dashboard")
    with gr.Tabs():
        with gr.TabItem("Predict & Save"):
            with gr.Row():
                with gr.Column(scale=3):
                    desc = gr.Textbox(label="Description", placeholder="e.g. ordered burger from swiggy.", lines=2)
                    amt = gr.Number(label="Amount (optional)", value=None)
                    thr = gr.Slider(label="Confidence threshold (>= accepted)", minimum=0.1, maximum=0.99, value=0.60, step=0.01)
                    with gr.Row():
                        predict_btn = gr.Button("Predict")
                        save_btn = gr.Button("Save")
                with gr.Column(scale=2):
                    out_dec = gr.Textbox(label="Decision", interactive=False)
                    out_top1 = gr.Textbox(label="Top-1 (label (prob))", interactive=False)
                    out_top2 = gr.Textbox(label="Top-2", interactive=False)
                    out_top3 = gr.Textbox(label="Top-3", interactive=False)
                    out_conf = gr.Number(label="Top-1 confidence", interactive=False)
                    corrected = gr.Dropdown(label="Corrected Label (optional)", choices=label_names, value=None)
                    status = gr.Textbox(label="Status / Errors", interactive=False)
            state = gr.State({})
            predict_btn.click(fn=predict_action, inputs=[desc, amt, thr], outputs=[out_dec, out_top1, out_top2, out_top3, out_conf, status, state])
            save_btn.click(fn=save_action, inputs=[corrected, state], outputs=[status, gr.JSON()])

        with gr.TabItem("Dashboard"):
            with gr.Row():
                refresh = gr.Button("Refresh Dashboard")
                download_log = gr.Button("Download log CSV")
            with gr.Row():
                month_plot = gr.Plot()
                cat_plot = gr.Plot()
            with gr.Row():
                heat_plot = gr.Plot()
            table = gr.Dataframe(headers=["timestamp","description","amount","category"], interactive=False)
            info = gr.Markdown("")
            # initial load
            def refresh_fn():
                fig_month, fig_cat, fig_heat, table_df = build_dashboard()
                info_txt = f"Loaded rows: {len(load_log_df())}. Showing recent rows in table."
                return fig_month, fig_cat, fig_heat, table_df, info_txt
            refresh.click(fn=refresh_fn, inputs=None, outputs=[month_plot, cat_plot, heat_plot, table, info])
            # download handler
            def download_fn():
                if LOG_PATH.exists():
                    return str(LOG_PATH)
                return "No log file."
            download_log.click(fn=download_fn, inputs=None, outputs=info)

    gr.Markdown("Saved entries file: `/content/drive/MyDrive/expense_entries_log.csv`")

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://339997b182bc1cecf9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
